# 🧹 Data Preprocessing Pipeline (Original Labels Preserved)
**Project**: YouTube Comments Sentiment Analysis  
**Labels Preserved**: `Negative` (0), `Neutral` (1), `Positive` (2)  
**Objective**: Clean raw YouTube comments using emoji demojization, YouTube slang normalization, negation-preserving stopword removal, and morphological lemmatization without changing original dataset labels.

In [1]:
# 1. Environment Setup & Dependency Loading
import os
import re
import emoji
import pandas as pd
import numpy as np
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK corpora silently
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Dependencies and NLTK resources initialized successfully!")

Dependencies and NLTK resources initialized successfully!


In [2]:
# 2. Text Cleaning & Normalization Functions
lemmatizer = WordNetLemmatizer()

# Preserve negation words in stopwords (crucial for sentiment polarity)
DEFAULT_STOPWORDS = set(stopwords.words('english'))
NEGATION_WORDS = {
    'no', 'not', 'nor', 'neither', 'never', 'none', "n't", 
    "don't", "doesn't", "didn't", "won't", "wouldn't", 
    "can't", "couldn't", "shouldn't", "isn't", "aren't", "wasn't", "weren't"
}
FILTERED_STOPWORDS = DEFAULT_STOPWORDS - NEGATION_WORDS

# YouTube & Social Media Slang Dictionary
SLANG_MAP = {
    "u": "you", "ur": "your", "r": "are", "b4": "before", "gr8": "great",
    "imo": "in my opinion", "imho": "in my humble opinion", "tbh": "to be honest",
    "lol": "laughing out loud", "rofl": "rolling on the floor laughing",
    "smh": "shaking my head", "nvm": "nevermind", "idk": "i do not know",
    "btw": "by the way", "omg": "oh my god", "pls": "please", "plz": "please",
    "w/": "with", "w/o": "without", "aka": "also known as", "fyi": "for your information"
}

def clean_comment_text(text: str) -> str:
    """
    Cleans raw comment text:
    - Demojizes emojis into descriptive text tokens (e.g. 😊 -> :smiling_face_with_smiling_eyes:)
    - Lowercases text
    - Strips hyperlinks and HTML tags
    - Expands internet slang
    - Strips non-alphanumeric noise while retaining underscores (used in demojized emojis)
    - Applies lemmatization while preserving sentiment-critical negation tokens
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Demojize emojis
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 2. Lowercase
    text = text.lower()
    
    # 3. Strip URLs & HTML tags
    text = re.sub(r'http\S+|www\S+|<.*?>', '', text)
    
    # 4. Expand slang
    words = text.split()
    words = [SLANG_MAP.get(w, w) for w in words]
    text = " ".join(words)
    
    # 5. Remove non-alphanumeric chars (retain underscores)
    text = re.sub(r'[^a-zA-Z0-9_\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 6. Lemmatize while preserving negation tokens
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in FILTERED_STOPWORDS]
    
    return " ".join(tokens)

print("Preprocessing pipeline compiled.")

Preprocessing pipeline compiled.


In [3]:
# 3. Load Dataset & Execute Preprocessing
KAGGLE_PATH = "/kaggle/input/datasets/amaanpoonawala/youtube-comments-sentiment-dataset/youtube_comments_cleaned.csv"
LOCAL_PATH = "youtube_comments_cleaned.csv"
input_path = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

print(f"Loading raw dataset from: {input_path}")
df = pd.read_csv(input_path)

# Drop missing values
df = df.dropna(subset=['CommentText', 'Sentiment']).reset_index(drop=True)
df['CommentText'] = df['CommentText'].astype(str)

# -----------------------------------------------------------------
# STRICTLY KEEP ORIGINAL LABELS: Negative (0), Neutral (1), Positive (2)
# -----------------------------------------------------------------
label_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df['label'] = df['Sentiment'].map(label_mapping)

df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print("Applying text cleaning pipeline...")
df['cleaned_comment'] = df['CommentText'].apply(clean_comment_text)

# Drop rows that became empty post-cleaning
df = df[df['cleaned_comment'].str.strip() != ""].reset_index(drop=True)

print("\n=== Class Distribution (Original Labels Kept Intact) ===")
dist = df['Sentiment'].value_counts()
for s in ['Negative', 'Neutral', 'Positive']:
    count = dist.get(s, 0)
    pct = (count / len(df)) * 100
    print(f"  • {s:<8}: {count:,} samples ({pct:.2f}%)")

Loading raw dataset from: /kaggle/input/datasets/amaanpoonawala/youtube-comments-sentiment-dataset/youtube_comments_cleaned.csv
Applying text cleaning pipeline...

=== Class Distribution (Original Labels Kept Intact) ===
  • Negative: 342,188 samples (33.54%)
  • Neutral : 336,816 samples (33.02%)
  • Positive: 341,165 samples (33.44%)


In [4]:
# 4. Save Preprocessed Dataset to CSV
output_filename = "youtube_comments_preprocessed.csv"
df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"Successfully exported preprocessed dataset to: {os.path.abspath(output_filename)}")
print(f"Total processed samples: {len(df):,}")
df[['CommentText', 'cleaned_comment', 'Sentiment', 'label']].head()

Successfully exported preprocessed dataset to: /kaggle/working/youtube_comments_preprocessed.csv
Total processed samples: 1,020,169


,CommentText,cleaned_comment,Sentiment,label
0,Anyone know what movie this is?,anyone know movie,Neutral,1
1,The fact they're holding each other back while...,fact theyre holding back equally aggressive fa...,Positive,2
2,waiting next video will be?,waiting next video,Neutral,1
3,Thanks for the great video.\n\nI don't underst...,thanks great video dont understand db continue...,Neutral,1
4,Good person helping good people.\nThis is how ...,good person helping good people america except...,Positive,2
